# Project Learning Guide: SOFR Multi-Curve Engine

This notebook is a guided path from zero familiarity to confident modification of this project. It sits before the existing lab notebooks: use it to learn the architecture, the Python patterns, the quantitative finance ideas, and the practical workflow for changing the code safely.

The core idea of the repo is simple but powerful: load a market snapshot, build an OIS discount curve, build a SOFR projection curve from futures and swaps, then use those curves for pricing, risk, diagnostics, and export.

## How To Use This Notebook

Run the cells from top to bottom once. After that, come back and use individual sections as references.

Suggested rhythm:

1. Read the explanation.
2. Run the code cell.
3. Change one small value.
4. Predict the result before re-running.
5. Compare your prediction with the output.

This is how you build project intuition. You are not just memorizing modules; you are learning the pressure points of the system.

## 0. Setup

This setup cell works whether the notebook is launched from the project root or from the `docs/` directory. It discovers the repo root by looking for `src/` and `data/`.

In [ ]:
from pathlib import Path
import json
import sys
from pprint import pprint

import numpy as np
import pandas as pd

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next(path for path in candidates if (path / 'src').exists() and (path / 'data').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.8f}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')

## 1. The One-Screen Architecture

Keep this map in your head:

`CSV/source system -> DataSource -> MarketData -> DiscountCurve + ProjectionCurve -> pricing -> risk/export/joint model`

The important design boundary is now explicit: raw inputs are handled by an object-oriented data input layer in `src/data_input.py`. After that boundary, the engine works with normalized domain objects such as `FuturesQuote`, `SwapQuote`, and `MarketData`.

That is the whole project. The modules are small because each one owns a specific idea.


In [ ]:
module_map = pd.DataFrame(
    [
        ('config.py', 'Load conventions and model settings into typed config objects.'),
        ('bootstrap.py', 'Load market data and build OIS/SOFR curves.'),
        ('curves.py', 'Represent discount curves and forward rates.'),
        ('interpolation.py', 'Interpolate log discount factors.'),
        ('instruments.py', 'Represent futures, swaps, and cashflow periods.'),
        ('dates.py / calendars.py / daycount.py', 'Date math, business-day rolling, and accrual fractions.'),
        ('hw_model.py', 'Hull-White convexity adjustment formulas.'),
        ('pricers.py', 'Par swap rates and swap PV.'),
        ('risk.py', 'PV01, key-rate DV01, and scenario tables.'),
        ('export.py / cli.py', 'Validated JSON market snapshot export.'),
        ('libor_model.py / libor_pricers.py', 'Mercurio-style LIBOR extension layer.'),
    ],
    columns=['Module', 'Responsibility'],
)
module_map

## 2. Files, Data, and Outputs

A maintainable project usually separates source code, input data, generated outputs, tests, exploratory notebooks, and explanatory docs. This repo follows that pattern closely:

- `src/` contains the engine code.
- `src/data_input.py` owns CSV reading, schema validation, type coercion, and construction of `MarketData`.
- `data/` contains raw synthetic input files.
- `notebooks/` contains numbered runnable walkthroughs.
- `docs/` contains explanatory learning material and longer writeups.
- `outputs/` contains generated reports, figures, curves, and tables.
- `tests/` protects the financial calculations and the input boundary.


In [ ]:
folder_map = pd.DataFrame(
    [
        ('src/', 'Production code.'),
        ('data/market/', 'Synthetic market quotes: futures, swaps, OIS curve, LIBOR calibration.'),
        ('data/fixings/', 'Historical SOFR fixings.'),
        ('data/metadata/', 'Project conventions and configuration.'),
        ('tests/', 'Executable specification of expected behavior.'),
        ('notebooks/', 'Focused lab notebooks and demos.'),
        ('outputs/', 'Generated figures, tables, reports, and curve snapshots.'),
        ('docs/', 'Long-form documentation and this learning guide.'),
    ],
    columns=['Folder', 'Purpose'],
)
folder_map

In [ ]:
market_files = sorted((PROJECT_ROOT / 'data' / 'market').glob('*.csv'))
summary = []
for path in market_files:
    frame = pd.read_csv(path)
    summary.append((path.name, len(frame), ', '.join(frame.columns)))

pd.DataFrame(summary, columns=['File', 'Rows', 'Columns'])

## 3. Configuration as a Design Boundary

Configuration is where project assumptions become explicit: valuation date, calendar, day-count conventions, Hull-White parameters, and risk bump sizes.

Notice one implementation detail: the file is named `conventions.yaml`, but its content is valid JSON, and `src/config.py` reads it with `json.loads`. That works because JSON is a strict subset of YAML, but if you later write actual YAML syntax, the current loader will fail.

In [ ]:
from src.config import load_engine_config

config_path = PROJECT_ROOT / 'data' / 'metadata' / 'conventions.yaml'
raw_config = json.loads(config_path.read_text(encoding='utf-8'))
pprint(raw_config)

config = load_engine_config(PROJECT_ROOT)
config

### Python Concept: Dataclasses

The config objects are `@dataclass(frozen=True)`. A dataclass gives you a compact class for structured data. `frozen=True` makes instances immutable after creation. This is useful for settings because it prevents accidental mutation halfway through a curve build.

In [ ]:
print(type(config.market))
print(config.market.valuation_date)

try:
    config.market.calendar = 'NEW_CALENDAR'
except Exception as exc:
    print(type(exc).__name__, str(exc))

## 4. Loading Market Data

`load_market_data()` is the public project entrance for the market snapshot, but it is intentionally thin. The real input work is delegated to a data source object.

By default, the project uses `CsvMarketDataSource(config.data_dir)`. That object reads the CSV files, checks required columns, rejects malformed cells, normalizes dates and numbers, and converts rows into domain objects such as `FuturesQuote` and `SwapQuote`.

This is the main reason the rest of the engine can use less defensive parsing: `bootstrap.py`, `curves.py`, and `pricers.py` should mostly receive already-clean inputs. If another source appears later, for example Bloomberg, Excel, SQL, or an API, it can provide the same `load(config)` behavior and return a `MarketData` object.


In [ ]:
from src.bootstrap import load_market_data
from src.config import load_engine_config
from src.data_input import CsvMarketDataSource, MarketData

config = load_engine_config(PROJECT_ROOT)
source = CsvMarketDataSource(config.data_dir)
market = load_market_data(PROJECT_ROOT, data_source=source)

print(type(market))
print(isinstance(market, MarketData))
print(f'1m futures: {len(market.futures_1m)}')
print(f'3m futures: {len(market.futures_3m)}')
print(f'swaps: {len(market.swaps)}')
print(f'OIS curve rows: {len(market.ois_curve)}')

market.ois_curve.head()


In [ ]:
# Mini pandas lab: inspect how parse_dates changes the dtype.
ois_path = PROJECT_ROOT / 'data' / 'market' / 'ois_curve.csv'
plain = pd.read_csv(ois_path)
parsed = pd.read_csv(ois_path, parse_dates=['end_date'])

print('Without parse_dates:', plain.dtypes.to_dict())
print('With parse_dates:', parsed.dtypes.to_dict())

parsed['python_date'] = parsed['end_date'].dt.date
parsed.head()

## 5. Dates, Day Counts, and Schedules

Rates code is mostly date code with finance attached. Before you touch pricing logic, understand how this repo builds periods and accrual factors.

Important ideas:

- a date schedule defines coupon boundaries;
- a business-day convention adjusts dates that fall on weekends;
- a day-count convention turns two dates into a year fraction;
- the year fraction controls interest accrual.

In [ ]:
from src.daycount import yearfrac
from src.instruments import build_periods

swap = market.swaps[0]
periods = build_periods(
    swap.start_date,
    swap.end_date,
    swap.pay_freq,
    swap.day_count,
    calendar=market.config.market.calendar,
    roll=market.config.market.business_day_roll,
)

period_table = pd.DataFrame([p.__dict__ for p in periods])
period_table.head(), period_table.tail()

In [ ]:
examples = [
    ('ACT/360', yearfrac(swap.start_date, swap.end_date, 'ACT/360')),
    ('ACT/365F', yearfrac(swap.start_date, swap.end_date, 'ACT/365F')),
    ('30/360', yearfrac(swap.start_date, swap.end_date, '30/360')),
]
pd.DataFrame(examples, columns=['Convention', 'Year Fraction'])

## 6. Curve Objects and Log-DF Interpolation

The central abstraction is `DiscountCurve`. It stores pillar dates and discount factors, then provides methods for discount factors, zero rates, forward rates, and bumped curves.

This project interpolates on log discount factors. That choice helps preserve positive discount factors between pillars and usually produces smoother continuously compounded zero-rate behavior than direct DF interpolation.

In [ ]:
from src.bootstrap import build_discount_curve

discount_curve = build_discount_curve(market.config.market.valuation_date, market.ois_curve)

curve_nodes = pd.DataFrame([node.__dict__ for node in discount_curve.nodes()])
curve_nodes.head()

In [ ]:
sample_dates = [node.pillar_date for node in discount_curve.nodes()[:: max(1, len(discount_curve.nodes()) // 5)]]
rows = []
for dt in sample_dates:
    rows.append(
        {
            'date': dt,
            'df': discount_curve.df(dt),
            'zero_rate': discount_curve.zero_rate(dt),
        }
    )

pd.DataFrame(rows)

## 7. Bootstrapping: The Project Heartbeat

Bootstrapping means solving curve nodes one after another so that the curve reprices observed market instruments.

The sequence is:

1. Start projection curve at valuation date with DF = 1.
2. Strip monthly SOFR DFs from 1m futures.
3. Extend with 3m futures.
4. Use swaps to solve longer annual pillars.
5. Build a `DiscountCurve` from the solved projection discount factors.

This section runs the same sequence in pieces so you can see the curve growing.

In [ ]:
from src.bootstrap import (
    bootstrap_sofr_short_end,
    bootstrap_from_1m_futures,
    bootstrap_from_3m_futures,
    bootstrap_from_swaps,
)

valuation_date = market.config.market.valuation_date
a = market.config.model.mean_reversion
sigma = market.config.model.sigma

projection_dfs = bootstrap_sofr_short_end(valuation_date)
print('After seed:', len(projection_dfs), sorted(projection_dfs.items())[:3])

projection_dfs = bootstrap_from_1m_futures(market.futures_1m, valuation_date, a, sigma, projection_dfs)
print('After 1m futures:', len(projection_dfs))

projection_dfs = bootstrap_from_3m_futures(market.futures_3m, valuation_date, a, sigma, projection_dfs)
print('After 3m futures:', len(projection_dfs))

projection_dfs = bootstrap_from_swaps(
    market.swaps,
    discount_curve,
    projection_dfs,
    calendar=market.config.market.calendar,
    roll=market.config.market.business_day_roll,
)
print('After swaps:', len(projection_dfs))

pd.DataFrame(sorted(projection_dfs.items()), columns=['pillar_date', 'projection_df']).tail()

### Python Concept: Why `dict[date, float]` Works Here

During bootstrapping, the code uses a dictionary from pillar date to discount factor. That is a good temporary data structure because each newly solved date becomes available for later instruments.

Later, the dictionary is sorted and converted into a curve object. This is a common design move: use a flexible structure while solving, then convert to a stricter domain object when the result is complete.

In [ ]:
first_future = market.futures_1m[0]
print(first_future)
print('Start date known before first future?', first_future.start_date in bootstrap_sofr_short_end(valuation_date))

one_step = bootstrap_from_1m_futures([first_future], valuation_date, a, sigma)
pd.DataFrame(sorted(one_step.items()), columns=['date', 'df'])

## 8. Hull-White Convexity Adjustments

Futures rates are not exactly forward rates because futures settle daily and are affected by convexity. The project uses a one-factor Hull-White layer with constant mean reversion `a` and constant volatility `sigma`.

When `sigma = 0`, the convexity adjustment disappears. That is an important sanity check and is covered by tests.

In [ ]:
from src.hw_model import convexity_1m, U_j_const_sigma
from src.daycount import yearfrac

rows = []
for quote in market.futures_1m[:6]:
    t_start = yearfrac(valuation_date, quote.start_date, 'ACT/365F')
    t_end = yearfrac(valuation_date, quote.end_date, 'ACT/365F')
    rows.append(
        {
            'contract': quote.contract_code,
            'implied_rate': quote.implied_rate,
            'convexity_sigma_1pct': convexity_1m(a, 0.01, t_start, t_end),
            'convexity_sigma_0': convexity_1m(a, 0.0, t_start, t_end),
        }
    )

pd.DataFrame(rows)

## 9. Build the Full Curves

`build_full_curves()` is the high-level API. Most users of this package should call this rather than manually running each bootstrap step.

The returned `CurveBuildResult` contains config, discount curve, projection curve, sigma, repricing tables, and diagnostics.

In [ ]:
from src.bootstrap import build_full_curves

result = build_full_curves(PROJECT_ROOT)

print(type(result))
print('sigma:', result.sigma)
pprint(result.diagnostics)

result.futures_repricing.head(), result.swap_repricing

In [ ]:
curves = pd.DataFrame(
    {
        'time': result.projection_curve.times,
        'projection_df': result.projection_curve.dfs,
    }
)
discount = pd.DataFrame(
    {
        'time': result.discount_curve.times,
        'discount_df': result.discount_curve.dfs,
    }
)

ax = curves.plot(x='time', y='projection_df', marker='o', figsize=(9, 4), title='Projection vs Discount Curves')
discount.plot(x='time', y='discount_df', marker='x', ax=ax)
ax.set_xlabel('Years from valuation date')
ax.set_ylabel('Discount factor')

## 10. Pricing a Swap

A fixed-vs-SOFR swap compares two legs:

- fixed leg: fixed_rate times accrual times OIS discount factor;
- floating leg: projected SOFR forward times accrual times OIS discount factor.

The important multi-curve design choice is this: projection comes from the SOFR projection curve, while discounting comes from the collateral/OIS discount curve.

In [ ]:
from src.pricers import par_swap_rate, pv_fixed_leg, pv_float_leg, pv_swap

quote = market.swaps[-1]
periods = build_periods(
    quote.start_date,
    quote.end_date,
    quote.pay_freq,
    quote.day_count,
    calendar=market.config.market.calendar,
    roll=market.config.market.business_day_roll,
)

notional = 100_000_000
fixed_pv = pv_fixed_leg(notional, quote.fixed_rate, periods, result.discount_curve)
float_pv = pv_float_leg(notional, periods, result.discount_curve, result.projection_curve)
swap_pv = pv_swap(notional, quote.fixed_rate, periods, result.discount_curve, result.projection_curve)
model_par_rate = par_swap_rate(result.discount_curve, result.projection_curve, periods)

pd.DataFrame(
    [
        ('market fixed rate', quote.fixed_rate),
        ('model par rate', model_par_rate),
        ('fixed leg PV', fixed_pv),
        ('floating leg PV', float_pv),
        ('swap PV', swap_pv),
    ],
    columns=['Measure', 'Value'],
)

## 11. Risk: PV01, Key-Rate DV01, and Scenarios

Risk code answers the question: how does valuation change if curves move?

This repo uses bump-and-revalue methods. That is not the fastest possible approach, but it is clear, robust, and easy to test.

In [ ]:
from src.risk import pv01, key_rate_dv01, run_scenarios

parallel_pv01 = pv01(notional, quote.fixed_rate, periods, result.discount_curve, result.projection_curve)
key_rates = key_rate_dv01(
    notional,
    quote.fixed_rate,
    periods,
    result.discount_curve,
    result.projection_curve,
    list(market.config.risk.key_rates_years),
)

print('PV01:', parallel_pv01)
pd.DataFrame(key_rates.items(), columns=['Key Rate Year', 'DV01'])

In [ ]:
no_convexity = build_full_curves(PROJECT_ROOT, sigma_override=0.0)
scenarios = run_scenarios(
    notional,
    quote.fixed_rate,
    periods,
    result.discount_curve,
    result.projection_curve,
    alt_projection_curve=no_convexity.projection_curve,
)
scenarios

## 12. Export Boundary

A good engine should let downstream systems consume results without importing all of its internals. This project exports a validated market snapshot JSON with curve nodes, model parameters, and diagnostics.

That export file is a contract. If you change it, update tests and think about downstream users.

In [ ]:
from src.export import build_market_snapshot, validate_market_snapshot

snapshot = build_market_snapshot(result)
validate_market_snapshot(snapshot)

print(snapshot.keys())
print(snapshot['valuation_date'])
print(snapshot['projection_curve'].keys())
snapshot['projection_curve']['nodes'][:3]

## 13. Mercurio Joint-Model Extension

The main project is an OIS/SOFR curve engine. The Mercurio layer extends that with synthetic LIBOR calibration diagnostics:

- shifted-lognormal forward LIBOR volatility;
- LIBOR-OIS multiplicative basis;
- basis-implied Hull-White sigma;
- compact fallback and basis-swap pricing helpers.

Treat this as an extension layer that consumes the curve build rather than as the center of the repo.

In [ ]:
from src.libor_model import build_joint_model_calibration

joint = build_joint_model_calibration(PROJECT_ROOT)
pprint(joint.diagnostics)
joint.calibration_table.head()

## 14. Tests Are Your Design Spec

Before modifying behavior, read the tests. They tell you what the project promises to preserve.

Important invariants:

- bootstrapped futures reprice almost exactly;
- swap repricing errors stay tiny;
- discount factors are positive;
- curves are monotone decreasing;
- zero-sigma convexity adjustments vanish;
- exported snapshots reject invalid discount factors.

In [ ]:
test_files = sorted((PROJECT_ROOT / 'tests').glob('test_*.py'))
rows = []
for path in test_files:
    text = path.read_text(encoding='utf-8')
    test_names = [line.strip().split('(')[0].replace('def ', '') for line in text.splitlines() if line.startswith('def test_')]
    rows.append((path.name, len(test_names), ', '.join(test_names)))

pd.DataFrame(rows, columns=['Test File', 'Count', 'Tests'])

If `pytest` is installed, run the full suite from the project root:

```powershell
python -m pytest -q
```

If it is not installed yet:

```powershell
python -m pip install -e ".[dev]"
python -m pytest -q
```

## 15. How To Modify the Project Safely

Use this loop for almost every change:

1. Identify the module that owns the behavior.
2. Read the related tests.
3. Write or update a focused test first when behavior changes.
4. Make the smallest code change that satisfies the behavior.
5. Run the narrow test file.
6. Run the full test suite.
7. Regenerate outputs only when the output contract intentionally changed.

Good first changes:

- add a new scenario in `src/risk.py`;
- add a new diagnostic field in `src/bootstrap.py`;
- add a new validation rule in `src/export.py`;
- extend calendar logic in `src/calendars.py`;
- make day-count usage in pricers configurable.

## 16. Mini Exercises

Use these exercises to become fluent without breaking the project.

### Exercise A: Change a model parameter without editing files

Run `build_full_curves(PROJECT_ROOT, sigma_override=0.0)` and compare diagnostics with the base result.

### Exercise B: Inspect one instrument end to end

Pick one swap quote. Build its periods, compute forwards for each period, discount each cashflow, and reproduce `pv_swap` manually.

### Exercise C: Add a scenario

In `src/risk.py`, add a scenario such as `Long-End Rally`, where rates after 5 years fall by 20 bp. Add a test that checks the scenario appears in the output.

### Exercise D: Add a diagnostic

Add `projection_min_df` and `discount_min_df` to `diagnostics_summary()`. Add a test in `tests/test_bootstrap.py`.

### Exercise E: Trace the active line

Start from `pd.read_csv(data_dir / 'market' / 'ois_curve.csv', parse_dates=['end_date'])`, then trace how those dates become OIS discount factors and later discount swap cashflows.

In [ ]:
# Exercise B starter: decompose one swap period by period.
cashflow_rows = []
for period in periods:
    forward = result.projection_curve.forward_rate(period.start_date, period.end_date, day_count='ACT/360')
    df = result.discount_curve.df(period.end_date)
    fixed_cf = notional * quote.fixed_rate * period.accrual_factor
    float_cf = notional * forward * period.accrual_factor
    cashflow_rows.append(
        {
            'start': period.start_date,
            'end': period.end_date,
            'accrual': period.accrual_factor,
            'forward': forward,
            'discount_df': df,
            'fixed_pv': fixed_cf * df,
            'float_pv': float_cf * df,
        }
    )

cashflows = pd.DataFrame(cashflow_rows)
cashflows.head()

In [ ]:
manual_fixed_pv = cashflows['fixed_pv'].sum()
manual_float_pv = cashflows['float_pv'].sum()
manual_swap_pv = manual_float_pv - manual_fixed_pv

pd.DataFrame(
    [
        ('manual fixed PV', manual_fixed_pv),
        ('function fixed PV', fixed_pv),
        ('manual float PV', manual_float_pv),
        ('function float PV', float_pv),
        ('manual swap PV', manual_swap_pv),
        ('function swap PV', swap_pv),
    ],
    columns=['Measure', 'Value'],
)

## 17. Reading Order After This Notebook

Once this guide feels comfortable, continue with the existing notebooks in this order:

1. `notebooks/00_project_demo.ipynb` for the full workflow.
2. `notebooks/01_data_check.ipynb` for data inspection.
3. `notebooks/02_curve_bootstrap.ipynb` for curve construction.
4. `notebooks/03_convexity_adjustment.ipynb` for Hull-White details.
5. `notebooks/04_swap_pricing_risk.ipynb` for pricing and risk.

For source code, read in this order:

1. `src/config.py`
2. `src/data_input.py`
3. `src/instruments.py`
4. `src/curves.py` and `src/interpolation.py`
5. `src/bootstrap.py`
6. `src/pricers.py`
7. `src/risk.py`
8. `src/export.py`
9. `src/libor_model.py`


## 18. What Mastery Looks Like

You are ready to modify this project when you can answer these questions without looking too much up:

- Which curve discounts cashflows and which curve projects SOFR?
- Why does bootstrapping need instruments sorted by maturity?
- What does `parse_dates` do in `pd.read_csv`?
- Why are configs and quote objects frozen dataclasses?
- What should happen when `sigma = 0`?
- Which tests protect repricing accuracy?
- What downstream contract does the snapshot export provide?

When those are natural, you are no longer just reading the repo. You are thinking in its design.